# CareerMatch AI — Career Matching

This notebook develops the career recommendation component of CareerMatch AI.
It uses the processed O*NET occupation profiles to match a candidate's
skills and experience with relevant occupations.

In [1]:
from pathlib import Path
import pandas as pd

Project_Root = Path.cwd().parent
PROCESSED_DATA_DIR = Project_Root / "data" / "processed"

profile_path = PROCESSED_DATA_DIR / "occupation_profiles.pkl"

occupation_profiles_df = pd.read_pickle(profile_path)

print("Dataset shape:", occupation_profiles_df.shape)
print("\nColumns:")
print(occupation_profiles_df.columns.tolist())

occupation_profiles_df.head()

Dataset shape: (1016, 6)

Columns:
['O*NET-SOC Code', 'Title', 'Description', 'Alternate Job Titles', 'Essential Skills', 'Software Skills']


,O*NET-SOC Code,Title,Description,Alternate Job Titles,Essential Skills,Software Skills
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,"[Aeronautics Commission Director, Agency Owner...","[{'Element Name': 'Active Learning', 'Importan...","[{'Workplace Example': 'Adobe Acrobat', 'Eleme..."
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",[CSR and Sustainability VP (Corporate Social R...,"[{'Element Name': 'Active Learning', 'Importan...","[{'Workplace Example': 'Adobe Acrobat', 'Eleme..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...","[Area Manager, Boards and Commissions Director...","[{'Element Name': 'Active Learning', 'Importan...","[{'Workplace Example': 'Act!', 'Element Name':..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes...","[Alderman, Assembly Member, Assembly Person, A...",NaN,"[{'Workplace Example': 'Adobe Acrobat', 'Eleme..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...","[Account Executive, Account Manager, Ad Accoun...","[{'Element Name': 'Active Learning', 'Importan...","[{'Workplace Example': 'Actuate BIRT', 'Elemen..."


In [2]:
data_scientist = occupation_profiles_df[
    occupation_profiles_df["O*NET-SOC Code"] == "15-2051.00"
].iloc[0]

print("Title:")
print(data_scientist["Title"])

print("\nEssential Skills:")
print(data_scientist["Essential Skills"][:3])

print("\nSoftware Skills:")
print(data_scientist["Software Skills"][:3])

Title:
Data Scientists

Essential Skills:
[{'Element Name': 'Active Learning', 'Importance': 3.62, 'Level': 4.38}, {'Element Name': 'Active Listening', 'Importance': 3.5, 'Level': 3.75}, {'Element Name': 'Critical Thinking', 'Importance': 4.25, 'Level': 4.62}]

Software Skills:
[{'Workplace Example': 'Alteryx software', 'Element Name': 'Business intelligence and data analysis software', 'Hot Technology': 'Y', 'In Demand': 'N'}, {'Workplace Example': 'Amazon Elastic Compute Cloud EC2', 'Element Name': 'Data base user interface and query software', 'Hot Technology': 'Y', 'In Demand': 'N'}, {'Workplace Example': 'Amazon Redshift', 'Element Name': 'Data base user interface and query software', 'Hot Technology': 'Y', 'In Demand': 'N'}]


## Create Occupation Matching Text

Each occupation profile contains structured information such as job titles,
essential skills, and software technologies. These fields are converted into
a single text representation for TF-IDF-based career matching.

In [3]:
def create_matching_text(row):
    alternate_titles = row["Alternate Job Titles"]
    essential_skills = row["Essential Skills"]
    software_skills = row["Software Skills"]

    # Handle occupations with missing data
    if not isinstance(alternate_titles, list):
        alternate_titles = []

    if not isinstance(essential_skills, list):
        essential_skills = []

    if not isinstance(software_skills, list):
        software_skills = []

    titles_text = " ".join(alternate_titles)

    skills_text = " ".join(
        skill["Element Name"]
        for skill in essential_skills
    )

    software_text = " ".join(
        software["Workplace Example"]
        for software in software_skills
    )

    matching_text = " ".join([
        row["Title"],
        row["Description"],
        titles_text,
        skills_text,
        software_text
    ])

    return matching_text

In [4]:
data_scientist_text = create_matching_text(data_scientist)

print(data_scientist_text[:1500])

Data Scientists Develop and implement a set of techniques or analytics applications to transform raw data into meaningful information using data-oriented programming languages and visualization software. Apply data mining, data modeling, natural language processing, and machine learning to extract and analyze information from large structured and unstructured datasets. Visualize, interpret, and report data findings. May create dynamic data reports. Analytics Consultant Applied Scientist Data Analyst Data Analytic Scientist Data Analytics Scientist Data Analytics Specialist Data Architect Data Consultant Data Economist Data Engineer Data Management Scientist Data Mining Analyst Data Modeler Data Quality Analyst Data Science Engineer Data Science Intern Data Specialist Machine Learning Data Scientist Machine Learning Engineer Machine Learning Scientist Machine Learning Software Engineer Marketing Data Scientist Psychometric Consultant Quantitative Methodologist Quantitative Researcher Re

In [5]:
occupation_profiles_df["Matching Text"] = (
    occupation_profiles_df.apply(
        create_matching_text,
        axis=1
    )
)

print("Number of occupation profiles:", len(occupation_profiles_df))

print(
    "Missing matching text:",
    occupation_profiles_df["Matching Text"].isnull().sum()
)

occupation_profiles_df[
    ["O*NET-SOC Code", "Title", "Matching Text"]
].head()

Number of occupation profiles: 1016
Missing matching text: 0


,O*NET-SOC Code,Title,Matching Text
0,11-1011.00,Chief Executives,Chief Executives Determine and formulate polic...
1,11-1011.03,Chief Sustainability Officers,Chief Sustainability Officers Communicate and ...
2,11-1021.00,General and Operations Managers,"General and Operations Managers Plan, direct, ..."
3,11-1031.00,Legislators,"Legislators Develop, introduce, or enact laws ..."
4,11-2011.00,Advertising and Promotions Managers,"Advertising and Promotions Managers Plan, dire..."


## Sample Candidate Profile

A sample candidate profile is used to test the career-matching pipeline
before integrating automatic résumé parsing and skill extraction.

In [6]:
candidate_profile = """
Python SQL machine learning data analysis pandas numpy
scikit-learn data visualization statistics
FastAPI natural language processing
"""

print(candidate_profile)


Python SQL machine learning data analysis pandas numpy
scikit-learn data visualization statistics
FastAPI natural language processing



In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [8]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

In [9]:
occupation_tfidf = tfidf_vectorizer.fit_transform(
    occupation_profiles_df["Matching Text"]
)

print("TF-IDF matrix shape:", occupation_tfidf.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (1016, 20372)
Vocabulary size: 20372


In [10]:
candidate_tfidf = tfidf_vectorizer.transform(
    [candidate_profile]
)

print("Candidate vector shape:", candidate_tfidf.shape)

Candidate vector shape: (1, 20372)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    candidate_tfidf,
    occupation_tfidf
).flatten()

print("Number of similarity scores:", len(similarity_scores))
print("Highest similarity score:", similarity_scores.max())

Number of similarity scores: 1016
Highest similarity score: 0.3212137931709196


In [12]:
top_5_indices = similarity_scores.argsort()[-5:][::-1]

top_5_matches = occupation_profiles_df.iloc[top_5_indices][
    ["O*NET-SOC Code", "Title"]
].copy()

top_5_matches["Similarity Score"] = similarity_scores[top_5_indices]

top_5_matches = top_5_matches.reset_index(drop=True)

print("Top 5 Career Recommendations:")
top_5_matches

Top 5 Career Recommendations:


,O*NET-SOC Code,Title,Similarity Score
0,15-2051.00,Data Scientists,0.321214
1,15-1243.01,Data Warehousing Specialists,0.166125
2,15-1243.00,Database Architects,0.159435
3,15-1221.00,Computer and Information Research Scientists,0.151571
4,43-9111.00,Statistical Assistants,0.138882


### Initial TF-IDF Baseline Result

The TF-IDF and cosine-similarity baseline successfully ranked all 1,016
O*NET occupations for the sample candidate profile. Data Scientists was
the highest-ranked occupation. These similarity values are used for
relative ranking and should not be interpreted as qualification
percentages.

The baseline results will later be compared with semantic matching using
Sentence-BERT.